In [ ]:
import zipfile
import pandas as pd
import os
from datetime import datetime
import json

# Path to your file
file_path = '../data/raw/KL_dataset.xlsx'

# ============================================
# 1. BASIC FILE INFORMATION
# ============================================
print("="*60)
print("FILE INFORMATION")
print("="*60)

file_stats = os.stat(file_path)
print(f"File name: {os.path.basename(file_path)}")
print(f"File size: {file_stats.st_size} bytes ({file_stats.st_size/1024/1024:.2f} MB)")
print(f"Created: {datetime.fromtimestamp(file_stats.st_ctime)}")
print(f"Modified: {datetime.fromtimestamp(file_stats.st_mtime)}")
print(f"Actual type: ZIP archive (PK signature detected)")

# ============================================
# 2. EXPLORE ZIP CONTENTS
# ============================================
print("\n" + "="*60)
print("ZIP ARCHIVE CONTENTS")
print("="*60)

with zipfile.ZipFile(file_path, 'r') as zip_ref:
    # Get all files info
    for file_info in zip_ref.infolist():
        print(f"\n📁 File: {file_info.filename}")
        print(f"   Size: {file_info.file_size} bytes ({file_info.file_size/1024:.2f} KB)")
        print(f"   Compressed: {file_info.compress_size} bytes")
        print(f"   Compression ratio: {(1 - file_info.compress_size/file_info.file_size)*100:.1f}%")
        print(f"   Modified: {datetime(*file_info.date_time)}")
    
    # List all files
    print("\n" + "-"*40)
    print("All files in archive:")
    for name in zip_ref.namelist():
        print(f"  • {name}")

# ============================================
# 3. EXTRACT AND ANALYZE THE ACTUAL DATA
# ============================================
print("\n" + "="*60)
print("DATA CONTENT ANALYSIS")
print("="*60)

# Extract and read the data
with zipfile.ZipFile(file_path, 'r') as zip_ref:
    # Find CSV files
    csv_files = [f for f in zip_ref.namelist() if f.endswith('.csv')]
    
    if csv_files:
        print(f"\nFound {len(csv_files)} CSV file(s):")
        
        for csv_file in csv_files:
            print(f"\n📊 Analyzing: {csv_file}")
            print("-"*40)
            
            # Read the CSV
            with zip_ref.open(csv_file) as f:
                # Try different encodings
                for encoding in ['utf-8', 'latin1', 'cp1252', 'iso-8859-1']:
                    try:
                        df = pd.read_csv(f, encoding=encoding)
                        print(f"✓ Successfully read with encoding: {encoding}")
                        break
                    except:
                        f.seek(0)  # Reset file pointer
                        continue
                
                # DATAFRAME METADATA
                print(f"\n📈 Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")
                
                print(f"\n📋 Column Information:")
                for col in df.columns:
                    print(f"  • {col}: {df[col].dtype} ({(df[col].count()/len(df))*100:.1f}% complete)")
                
                print(f"\n📊 Data Types Summary:")
                print(df.dtypes.value_counts())
                
                print(f"\n🔢 Memory Usage: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")
                
                print(f"\n👀 First 5 rows:")
                print(df.head())
                
                print(f"\n📑 Last 5 rows:")
                print(df.tail())
                
                print(f"\n📊 Basic Statistics (numeric columns):")
                print(df.describe())
                
                print(f"\n🔍 Missing Values:")
                missing = df.isnull().sum()
                missing = missing[missing > 0]
                if len(missing) > 0:
                    print(missing)
                else:
                    print("No missing values found!")
                
                print(f"\n🔄 Unique Values per Column:")
                for col in df.columns:
                    unique_count = df[col].nunique()
                    print(f"  • {col}: {unique_count} unique values")
                
                # Save metadata to file
                metadata = {
                    'file_name': os.path.basename(file_path),
                    'file_size_mb': file_stats.st_size/1024/1024,
                    'dataset_name': csv_file,
                    'shape': {'rows': df.shape[0], 'columns': df.shape[1]},
                    'columns': list(df.columns),
                    'dtypes': df.dtypes.astype(str).to_dict(),
                    'missing_values': df.isnull().sum().to_dict(),
                    'unique_counts': df.nunique().to_dict(),
                    'memory_usage_kb': df.memory_usage(deep=True).sum() / 1024
                }
                
                # Save metadata as JSON
                with open('../data/raw/dataset_metadata.json', 'w') as json_file:
                    json.dump(metadata, json_file, indent=2)
                print(f"\n💾 Metadata saved to: dataset_metadata.json")
                
                # Assign to variable for later use
                cbc_data = df
    else:
        print("No CSV files found in the archive")

# ============================================
# 4. CHECK FOR OTHER DATA FORMATS
# ============================================
print("\n" + "="*60)
print("ADDITIONAL FILES IN ARCHIVE")
print("="*60)

with zipfile.ZipFile(file_path, 'r') as zip_ref:
    # Check for Excel files
    excel_files = [f for f in zip_ref.namelist() if f.endswith(('.xlsx', '.xls'))]
    if excel_files:
        print(f"\n📊 Excel files found: {excel_files}")
    
    # Check for JSON files
    json_files = [f for f in zip_ref.namelist() if f.endswith('.json')]
    if json_files:
        print(f"\n📋 JSON files found: {json_files}")
    
    # Check for text files
    txt_files = [f for f in zip_ref.namelist() if f.endswith('.txt')]
    if txt_files:
        print(f"\n📝 Text files found: {txt_files}")

print("\n" + "="*60)
print("ANALYSIS COMPLETE!")
print("="*60)

In [ ]:
%pip install openpyxl

In [ ]:
import pandas as pd

# Read the Excel file
df = pd.read_excel('../data/raw/KL_dataset.xlsx')

# View your data
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst 5 rows:")
print(df.head())
print(f"\nLast 5 rows:")
print(df.tail())
print(f"\nData types:")
print(df.dtypes)
print(f"\nBasic info:")
df.info()

In [ ]:
# Informations générales : types de colonnes, valeurs non-nulles
df.info()

In [ ]:
# Statistiques descriptives
df.describe(include='all')

In [ ]:
# Liste des colonnes
print(df.columns.tolist())

In [ ]:
# Vérifier les valeurs manquantes
df.isnull().sum()

In [ ]:
# Vérifier les doublons
print("Doublons:", df.duplicated().sum())